# Construire votre Réseau de Neurones Profond : Étape par Étape

Dans ce notebook, vous implémenterez toutes les fonctions nécessaires à la construction d'un réseau de neurones profond avec autant de couches que vous le souhaitez.

- Dans ce notebook, vous implémenterez toutes les briques fondamentales d'un réseau de neurones profond.
- Dans le notebook suivant, vous utiliserez ces fonctions pour construire un réseau profond appliqué à la classification d'images.

**À la fin de ce workshop, vous serez capable de :**

- Utiliser des unités non linéaires comme ReLU pour améliorer votre modèle
- Construire un réseau de neurones plus profond (avec plus d'une couche cachée)
- Implémenter une classe de réseau de neurones simple d'utilisation

**Notation** :
- L'exposant $[l]$ désigne une quantité associée à la $l^{	ext{ème}}$ couche.
- L'exposant $(i)$ désigne une quantité associée au $i^{	ext{ème}}$ exemple.
- L'indice $i$ désigne le $i^{	ext{ème}}$ élément d'un vecteur.

**Note sur l'évaluateur automatique :** Ce notebook contient des cellules évaluées automatiquement. Évitez d'ajouter des `print` supplémentaires, de modifier les paramètres des fonctions, ou d'utiliser des variables globales dans les exercices.

Commençons !

## Table des matières
- [1 - Packages](#1)
- [2 - Plan du notebook](#2)
- [3 - Initialisation](#3)
 - [3.1 - Réseau de neurones à 2 couches](#3-1)
 - [Exercice 1 - initialize_parameters](#ex-1)
 - [3.2 - Réseau de neurones profond à L couches](#3-2)
 - [Exercice 2 - initialize_parameters_deep](#ex-2)
- [4 - Module de propagation avant](#4)
 - [4.1 - Propagation linéaire avant](#4-1)
 - [Exercice 3 - linear_forward](#ex-3)
 - [4.2 - Propagation linéaire-activation avant](#4-2)
 - [Exercice 4 - linear_activation_forward](#ex-4)
 - [4.3 - Modèle à L couches](#4-3)
 - [Exercice 5 - L_model_forward](#ex-5)
- [5 - Fonction de coût](#5)
 - [Exercice 6 - compute_cost](#ex-6)
- [6 - Module de rétropropagation](#6)
 - [6.1 - Propagation linéaire arrière](#6-1)
 - [Exercice 7 - linear_backward](#ex-7)
 - [6.2 - Propagation linéaire-activation arrière](#6-2)
 - [Exercice 8 - linear_activation_backward](#ex-8)
 - [6.3 - Rétropropagation du modèle à L couches](#6-3)
 - [Exercice 9 - L_model_backward](#ex-9)
 - [6.4 - Mise à jour des paramètres](#6-4)
 - [Exercice 10 - Mettre à jour les paramètres](#ex-10)

<a name='1'></a>
## 1 - Bibliothèques

Commencez par importer tous les packages dont vous aurez besoin dans ce notebook.

- [numpy](www.numpy.org) est le package principal pour le calcul scientifique avec Python.
- [matplotlib](http://matplotlib.org) est une bibliothèque pour tracer des graphiques en Python.
- `dnn_utils` fournit quelques fonctions utiles pour ce notebook.
- `testCases` fournit des cas de test pour vérifier la correction de vos fonctions.
- `np.random.seed(1)` est utilisé pour assurer la reproductibilité des résultats. Ne le modifiez pas.

In [ ]:
### v1.2

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from testCases import *
from dnn_utils import sigmoid, sigmoid_backward, relu, relu_backward
from public_tests import *

import copy
%matplotlib inline
plt.rcParams['figure.figsize'] = (5.0, 4.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

np.random.seed(1)

<a name='2'></a>
## 2 - Plan du notebook

Pour construire votre réseau de neurones, vous allez implémenter plusieurs "fonctions auxiliaires". Ces fonctions seront utilisées dans le prochain notebook pour construire un réseau à deux couches et un réseau à L couches.

Chaque fonction auxiliaire est accompagnée d'instructions détaillées. Voici un aperçu des étapes de ce notebook :

- Initialiser les paramètres d'un réseau à deux couches et d'un réseau à L couches
- Implémenter le module de propagation avant (en violet dans la figure ci-dessous)
 - Compléter la partie LINÉAIRE de la propagation avant d'une couche (calcul de $Z^{[l]}$).
 - La fonction ACTIVATION (relu/sigmoid) est fournie.
 - Combiner ces deux étapes dans une nouvelle fonction [LINÉAIRE->ACTIVATION].
 - Empiler [LINÉAIRE->RELU] L-1 fois et ajouter [LINÉAIRE->SIGMOID] à la fin. Cela donne la fonction L_model_forward.
- Calculer la perte
- Implémenter le module de rétropropagation (en rouge dans la figure ci-dessous)
 - Compléter la partie LINÉAIRE de la rétropropagation d'une couche
 - Le gradient de la fonction ACTIVATION est fourni (relu_backward/sigmoid_backward)
 - Combiner ces deux étapes dans une nouvelle fonction [LINÉAIRE->ACTIVATION] arrière
 - Empiler [LINÉAIRE->RELU] arrière L-1 fois et ajouter [LINÉAIRE->SIGMOID] arrière dans L_model_backward
- Mettre à jour les paramètres

<img src="images/final outline.png" style="width:800px;height:500px;">
<caption><center><b>Figure 1</b></center></caption><br>

**Note** :

Pour chaque fonction de propagation avant, il existe une fonction de rétropropagation correspondante. C'est pourquoi à chaque étape du module avant, vous stockerez certaines valeurs dans un cache. Ces valeurs mises en cache sont utiles pour calculer les gradients lors de la rétropropagation.

<a name='3'></a>
## 3 - Initialisation

Vous allez écrire deux fonctions auxiliaires pour initialiser les paramètres de votre modèle. La première fonction sera utilisée pour initialiser les paramètres d'un modèle à deux couches. La seconde généralise ce processus d'initialisation à $L$ couches.

<a name='3-1'></a>
### 3.1 - Réseau de neurones à 2 couches

<a name='ex-1'></a>
### Exercice 1 - initialize_parameters

Créez et initialisez les paramètres du réseau de neurones à 2 couches.

**Instructions :**

- La structure du modèle est : *LINÉAIRE -> RELU -> LINÉAIRE -> SIGMOÏDE*. - Utilisez cette initialisation aléatoire pour les matrices de poids : `np.random.randn(d0, d1, ..., dn) * 0.01` avec la forme correcte. La documentation pour [np.random.randn](https://numpy.org/doc/stable/reference/random/generated/numpy.random.randn.html)
- Utilisez l'initialisation à zéro pour les biais : `np.zeros(shape)`. La documentation pour [np.zeros](https://numpy.org/doc/stable/reference/generated/numpy.zeros.html)

In [ ]:
# FONCTION : initialize_parameters
def initialize_parameters(n_x, n_h, n_y):
    """
 Arguments :
 n_x -- taille de la couche d'entrée
 n_h -- taille de la couche cachée
 n_y -- taille de la couche de sortie
 
 Retourne :
 paramètres -- dictionnaire python contenant vos paramètres :
 W1 -- matrice de poids de forme (n_h, n_x)
 b1 -- vecteur de biais de forme (n_h, 1)
 W2 -- matrice de poids de forme (n_y, n_h)
 b2 -- vecteur de biais de forme (n_y, 1)
 """
    
    np.random.seed(1)
    
    #(≈ 4 lignes de code) # W1 = ... # b1 = ... # W2 = ... # b2 = ... # VOTRE CODE COMMENCE ICI 
    # VOTRE CODE SE TERMINE ICI parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2}
    
    return parameters    

In [ ]:
print("Test Case 1:\n")
parameters = initialize_parameters(3,2,1)

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_test_1(initialize_parameters)

print("\033[90m\nTest Case 2:\n")
parameters = initialize_parameters(4,3,2)

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_test_2(initialize_parameters)

***Sortie attendue***
```
Cas de test 1:

W1 = [[ 0.01624345 -0.00611756 -0.00528172]
 [-0.01072969 0.00865408 -0.02301539]]
b1 = [[0.]
 [0.]]
W2 = [[ 0.01744812 -0.00761207]]
b2 = [[0.]]
 All tests passed.

Cas de test 2:

W1 = [[ 0.01624345 -0.00611756 -0.00528172 -0.01072969]
 [ 0.00865408 -0.02301539 0.01744812 -0.00761207]
 [ 0.00319039 -0.0024937 0.01462108 -0.02060141]]
b1 = [[0.]
 [0.]
 [0.]]
W2 = [[-0.00322417 -0.00384054 0.01133769]
 [-0.01099891 -0.00172428 -0.00877858]]
b2 = [[0.]
 [0.]]
 All tests passed.
```

<a name='3-2'></a>
### 3.2 - Réseau de neurones profond à L couches

L'initialisation d'un réseau de neurones plus profond à L couches est plus complexe car il y a davantage de matrices de poids et de vecteurs de biais. Lorsque vous complétez la fonction `initialize_parameters_deep`, assurez-vous que les dimensions correspondent entre chaque couche. Rappelez-vous que $n^{[l]}$ est le nombre d'unités de la couche $l$. Par exemple, si la taille de votre entrée $X$ est $(12288, 209)$ (avec $m=209$ exemples), alors :

<table style="width:100%">
 <tr>
 <td> </td>  <td> <b>Forme de W</b> </td>  <td> <b>Forme de b</b> </td>  <td> <b>Activation</b> </td>
 <td> <b>Forme de l'Activation</b> </td>  <tr>
 <tr>
 <td> <b>couche 1</b> </td>  <td> $(n^{[1]},12288)$ </td>  <td> $(n^{[1]},1)$ </td>  <td> $Z^{[1]} = W^{[1]} X + b^{[1]} $ </td>  <td> $(n^{[1]},209)$ </td>  <tr>
 <tr>
 <td> <b>couche 2</b> </td>  <td> $(n^{[2]}, n^{[1]})$ </td>  <td> $(n^{[2]},1)$ </td>  <td>$Z^{[2]} = W^{[2]} A^{[1]} + b^{[2]}$ </td>  <td> $(n^{[2]}, 209)$ </td>  <tr>
 <tr>
 <td> $\vdots$ </td>  <td> $\vdots$ </td>  <td> $\vdots$ </td>  <td> $\vdots$</td>  <td> $\vdots$ </td>  <tr>  <tr>
 <td> <b>couche L-1</b> </td>  <td> $(n^{[L-1]}, n^{[L-2]})$ </td>  <td> $(n^{[L-1]}, 1)$ </td>  <td>$Z^{[L-1]} = W^{[L-1]} A^{[L-2]} + b^{[L-1]}$ </td>  <td> $(n^{[L-1]}, 209)$ </td>  <tr>
 <tr>
 <td> <b>couche L</b> </td>  <td> $(n^{[L]}, n^{[L-1]})$ </td>  <td> $(n^{[L]}, 1)$ </td>
 <td> $Z^{[L]} = W^{[L]} A^{[L-1]} + b^{[L]}$</td>
 <td> $(n^{[L]}, 209)$ </td>  <tr>
</table>

Rappelons que lorsqu'on calcule $W X + b$ en Python, le broadcasting est utilisé. Par exemple, si : 
$$ W = \begin{bmatrix}
 w_{00} & w_{01} & w_{02} \\
 w_{10} & w_{11} & w_{12} \\
 w_{20} & w_{21} & w_{22} \end{bmatrix}\;\;\; X = \begin{bmatrix}
 x_{00} & x_{01} & x_{02} \\
 x_{10} & x_{11} & x_{12} \\
 x_{20} & x_{21} & x_{22} \end{bmatrix} \;\;\; b =\begin{bmatrix}
 b_0 \\
 b_1 \\
 b_2
\end{bmatrix}\tag{2}$$

Alors $WX + b$ vaut :

$$ WX + b = \begin{bmatrix}
 (w_{00}x_{00} + w_{01}x_{10} + w_{02}x_{20}) + b_0 & (w_{00}x_{01} + w_{01}x_{11} + w_{02}x_{21}) + b_0 & \cdots \\
 (w_{10}x_{00} + w_{11}x_{10} + w_{12}x_{20}) + b_1 & (w_{10}x_{01} + w_{11}x_{11} + w_{12}x_{21}) + b_1 & \cdots \\
 (w_{20}x_{00} + w_{21}x_{10} + w_{22}x_{20}) + b_2 & (w_{20}x_{01} + w_{21}x_{11} + w_{22}x_{21}) + b_2 & \cdots
\end{bmatrix}\tag{3} $$

<a name='ex-2'></a>
### Exercice 2 - initialize_parameters_deep

Implémentez l'initialisation pour un réseau de neurones profond à L couches. 
**Instructions :**
- La structure du modèle est *[LINÉAIRE -> RELU] $ \times$ (L-1) -> LINÉAIRE -> SIGMOÏDE*. Elle comporte $L-1$ couches utilisant la fonction d'activation ReLU, suivies d'une couche de sortie avec une activation sigmoïde.
- Utilisez l'initialisation aléatoire pour les matrices de poids : `np.random.randn(d0, d1, ..., dn) * 0.01`.
- Utilisez l'initialisation à zéro pour les biais : `np.zeros(shape)`.
- Vous stockerez $n^{[l]}$, le nombre d'unités dans les différentes couches, dans une variable `layer_dims`. Par exemple, le `layer_dims` pour le modèle de classification de données planaires serait [2,4,1] : deux entrées, une couche cachée avec 4 unités, et une couche de sortie avec 1 unité. Cela signifie que `W1` aurait la forme (4,2), `b1` (4,1), `W2` (1,4) et `b2` (1,1). Vous allez maintenant généraliser cela à $L$ couches !
- Voici l'implémentation pour $L=1$ (réseau de neurones à une couche). Elle peut vous guider pour implémenter le cas général (réseau à L couches).
```python
 if L == 1:
    paramètres["W" + str(L)] = np.random.randn(layer_dims[1], layer_dims[0]) * 0.01
    paramètres["b" + str(L)] = np.zeros((layer_dims[1], 1))
```

In [ ]:
# FONCTION : initialize_parameters_deep
def initialize_parameters_deep(layer_dims):
    
    """ Arguments : layer_dims -- tableau python (list) contenant les dimensions de chaque couche du réseau Retourne : paramètres -- dictionnaire python contenant vos paramètres "W1", "b1", ..., "WL", "bL": Wl -- matrice de poids de forme (layer_dims[l], layer_dims[l-1]) bl -- vecteur de biais de forme (layer_dims[l], 1) """
    
    np.random.seed(3)
    parameters = {}
    L = len(layer_dims) # nombre de couches dans le réseau
    for l in range(1, L):
        #(≈ 2 lignes de code) # parameters['W' + str(l)] = ... # parameters['b' + str(l)] = ... # VOTRE CODE COMMENCE ICI 
        # VOTRE CODE SE TERMINE ICI assert(parameters['W' + str(l)].shape == (layer_dims[l], layer_dims[l - 1]))
        assert(parameters['b' + str(l)].shape == (layer_dims[l], 1))

        
    return parameters

In [ ]:
print("Test Case 1:\n")
parameters = initialize_parameters_deep([5,4,3])

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_deep_test_1(initialize_parameters_deep)

print("\033[90m\nTest Case 2:\n")
parameters = initialize_parameters_deep([4,3,2])

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))
initialize_parameters_deep_test_2(initialize_parameters_deep)

***Sortie attendue***
```
Cas de test 1:

W1 = [[ 0.01788628 0.0043651 0.00096497 -0.01863493 -0.00277388]
 [-0.00354759 -0.00082741 -0.00627001 -0.00043818 -0.00477218]
 [-0.01313865 0.00884622 0.00881318 0.01709573 0.00050034]
 [-0.00404677 -0.0054536 -0.01546477 0.00982367 -0.01101068]]
b1 = [[0.]
 [0.]
 [0.]
 [0.]]
W2 = [[-0.01185047 -0.0020565 0.01486148 0.00236716]
 [-0.01023785 -0.00712993 0.00625245 -0.00160513]
 [-0.00768836 -0.00230031 0.00745056 0.01976111]]
b2 = [[0.]
 [0.]
 [0.]]
 All tests passed.

Cas de test 2:

W1 = [[ 0.01788628 0.0043651 0.00096497 -0.01863493]
 [-0.00277388 -0.00354759 -0.00082741 -0.00627001]
 [-0.00043818 -0.00477218 -0.01313865 0.00884622]]
b1 = [[0.]
 [0.]
 [0.]]
W2 = [[ 0.00881318 0.01709573 0.00050034]
 [-0.00404677 -0.0054536 -0.01546477]]
b2 = [[0.]
 [0.]]
 All tests passed.
```

<a name='4'></a>
## 4 - Module de propagation avant

<a name='4-1'></a>
### 4.1 - Propagation linéaire avant 
Maintenant que vous avez initialisé vos paramètres, vous pouvez passer au module de propagation avant. Commencez par implémenter quelques fonctions de base que vous pourrez réutiliser lors de l'implémentation du modèle. Vous allez compléter trois fonctions dans cet ordre :

- LINÉAIRE
- LINÉAIRE -> ACTIVATION, où ACTIVATION sera soit ReLU soit Sigmoïde. - [LINÉAIRE -> RELU] $\times$ (L-1) -> LINÉAIRE -> SIGMOÏDE (modèle complet)

Le module linéaire avant (vectorisé sur tous les exemples) calcule les équations suivantes :

$$Z^{[l]} = W^{[l]}A^{[l-1]} +b^{[l]}\tag{4}$$

où $A^{[0]} = X$. 
<a name='ex-3'></a>
### Exercice 3 - linear_forward 
Construisez la partie linéaire de la propagation avant.

**Rappel :**
La représentation mathématique de cette unité est $Z^{[l]} = W^{[l]}A^{[l-1]} +b^{[l]}$. Vous pouvez également utiliser `np.dot()`. Si vos dimensions ne correspondent pas, afficher `W.shape` peut vous aider.

In [ ]:
# FONCTION : linear_forward
def linear_forward(A, W, b):
    """ Implémente la partie linéaire de la propagation avant d'une couche. Arguments : A -- activations de la couche précédente (ou données d'entrée) : (taille de la couche précédente, nombre d'exemples) W -- matrice de poids : tableau numpy de forme (taille de la couche actuelle, taille de la couche précédente) b -- vecteur de biais, tableau numpy de forme (taille de la couche actuelle, 1) Retourne : Z -- l'entrée de la fonction d'activation, aussi appelée paramètre de pré-activation cache -- un tuple python contenant "A", "W" et "b" ; stocké pour calculer la passe arrière efficacement """
    
    #(≈ 1 ligne de code) # Z = ... # VOTRE CODE COMMENCE ICI # VOTRE CODE SE TERMINE ICI cache = (A, W, b) 
    return Z, cache

In [ ]:
t_A, t_W, t_b = linear_forward_test_case()
t_Z, t_linear_cache = linear_forward(t_A, t_W, t_b)
print("Z = " + str(t_Z))

linear_forward_test(linear_forward)

***Sortie attendue***
```
Z = [[ 3.26295337 -1.23429987]]
```

<a name='4-2'></a>
### 4.2 - Propagation linéaire-activation avant

Dans ce notebook, vous allez utiliser deux fonctions d'activation :

- **Sigmoid** : $\sigma(Z) = \sigma(W A + b) = \frac{1}{ 1 + e^{-(W A + b)}}$. La fonction `sigmoid` vous est fournie et retourne **deux** éléments : la valeur d'activation `"a"` et un `"cache"` contenant `"Z"` (qui sera passé à la fonction arrière correspondante). Pour l'utiliser : ``` python
A, activation_cache = sigmoid(Z)
```

- **ReLU** : La formule mathématique de ReLU est $A = RELU(Z) = max(0, Z)$. La fonction `relu` vous est fournie et retourne **deux** éléments : la valeur d'activation `"A"` et un `"cache"` contenant `"Z"` (qui sera passé à la fonction arrière correspondante). Pour l'utiliser :
``` python
A, activation_cache = relu(Z)
```

Pour plus de simplicité, vous allez regrouper deux fonctions (Linéaire et Activation) en une seule fonction (LINÉAIRE->ACTIVATION). Vous implémenterez donc une fonction qui effectue l'étape de propagation avant LINÉAIRE, suivie de l'étape d'ACTIVATION.

<a name='ex-4'></a>
### Exercice 4 - linear_activation_forward

Implémentez la propagation avant pour la couche *LINÉAIRE->ACTIVATION*. La relation mathématique est : $A^{[l]} = g(Z^{[l]}) = g(W^{[l]}A^{[l-1]} +b^{[l]})$ où l'activation "g" peut être sigmoid() ou relu(). Utilisez `linear_forward()` et la fonction d'activation correcte.

In [ ]:
# FONCTION : linear_activation_forward
def linear_activation_forward(A_prev, W, b, activation):
    """
 Implémente la propagation avant pour la couche LINÉAIRE->ACTIVATION

 Arguments :
 A_prev -- activations de la couche précédente (ou données d'entrée) : (taille de la couche précédente, nombre d'exemples)
 W -- matrice de poids : tableau numpy de forme (taille de la couche actuelle, taille de la couche précédente)
 b -- vecteur de biais, tableau numpy de forme (taille de la couche actuelle, 1)
 activation -- la fonction d'activation à utiliser dans cette couche, sous forme de chaîne de caractères : "sigmoid" ou "relu"

 Retourne :
 A -- la sortie de la fonction d'activation, aussi appelée valeur post-activation
 cache -- un tuple python contenant "linear_cache" et "activation_cache" ;
 stocké pour calculer la passe arrière efficacement
 """
    
    if activation == "sigmoid":
        #(≈ 2 lignes de code) # Z, linear_cache = ... # A, activation_cache = ... # VOTRE CODE COMMENCE ICI 
        # VOTRE CODE SE TERMINE ICI
    elif activation == "relu":
        #(≈ 2 lignes de code) # Z, linear_cache = ... # A, activation_cache = ... # VOTRE CODE COMMENCE ICI
        # VOTRE CODE SE TERMINE ICI cache = (linear_cache, activation_cache)
    return A, cache

In [ ]:
t_A_prev, t_W, t_b = linear_activation_forward_test_case()

t_A, t_linear_activation_cache = linear_activation_forward(t_A_prev, t_W, t_b, activation = "sigmoid")
print("Avec sigmoid : A = " + str(t_A))

t_A, t_linear_activation_cache = linear_activation_forward(t_A_prev, t_W, t_b, activation = "relu")
print("Avec ReLU : A = " + str(t_A))

linear_activation_forward_test(linear_activation_forward)

***Sortie attendue***
```
Avec sigmoid : A = [[0.96890023 0.11013289]]
Avec ReLU : A = [[3.43896131 0. ]]
```

**Note :** En apprentissage profond, le calcul "[LINÉAIRE->ACTIVATION]" est compté comme une seule couche dans le réseau de neurones, et non deux couches.

<a name='4-3'></a>
### 4.3 - Modèle à L couches 
Pour encore plus de simplicité lors de l'implémentation du réseau à $L$ couches, vous aurez besoin d'une fonction qui répète la précédente (`linear_activation_forward` avec RELU) $L-1$ fois, puis enchaîne avec un `linear_activation_forward` utilisant SIGMOID.

<img src="images/model_architecture_kiank.png" style="width:600px;height:300px;">
<caption><center> <b>Figure 2</b> : *[LINÉAIRE -> RELU] $\times$ (L-1) -> LINÉAIRE -> SIGMOÏDE* modèle</center></caption><br>

<a name='ex-5'></a>
### Exercice 5 - L_model_forward

Implémentez la propagation avant pour le modèle ci-dessus.

**Instructions :** Dans le code ci-dessous, la variable `AL` désigne $A^{[L]} = \sigma(Z^{[L]}) = \sigma(W^{[L]} A^{[L-1]} + b^{[L]})$. (Parfois aussi appelé `Yhat`, c'est-à-dire $\hat{Y}$.) 
**Indices :**
- Utilisez les fonctions implémentées précédemment - Utilisez une boucle `for` pour répéter [LINÉAIRE->RELU] (L-1) fois
- N'oubliez pas de garder trace des caches dans la liste "caches". Pour ajouter une nouvelle valeur `c` à une `list`, vous pouvez utiliser `list.append(c)`.

In [ ]:
# FONCTION : L_model_forward
def L_model_forward(X, parameters):
    """ Implémente la propagation avant pour [LINÉAIRE->RELU]*(L-1)->LINÉAIRE->SIGMOÏDE Arguments : X -- données, tableau numpy de forme (taille d'entrée, nombre d'exemples) paramètres -- sortie de initialize_parameters_deep() Retourne : AL -- valeur d'activation de la couche de sortie (dernière couche) caches -- liste de caches contenant : chaque cache de linear_activation_forward() (il y en a L, indexés de 0 à L-1) """

    caches = []
    A = X
    L = len(parameters) // 2                  # nombre de couches dans le réseau de neurones
    
    # Implémenter [LINÉAIRE -> RELU]*(L-1). Ajouter "cache" à la liste "caches". # La boucle for commence à 1 car la couche 0 est l'entrée for l in range(1, L): A_prev = A 
        #(≈ 2 lignes de code) # A, cache = ... # caches ... # VOTRE CODE COMMENCE ICI # VOTRE CODE SE TERMINE ICI # Implémenter LINÉAIRE -> SIGMOÏDE. Ajouter "cache" à la liste "caches". #(≈ 2 lignes de code) # AL, cache = ... # caches ... # VOTRE CODE COMMENCE ICI # VOTRE CODE SE TERMINE ICI
    return AL, caches

In [ ]:
t_X, t_parameters = L_model_forward_test_case_2hidden()
t_AL, t_caches = L_model_forward(t_X, t_parameters)

print("AL = " + str(t_AL))

L_model_forward_test(L_model_forward)

***Sortie attendue***
```
AL = [[0.03921668 0.70498921 0.19734387 0.04728177]]
```

**Génial !** Vous venez d'implémenter une propagation avant complète qui prend en entrée X et produit un vecteur ligne $A^{[L]}$ contenant vos prédictions. Elle enregistre également toutes les valeurs intermédiaires dans "caches". En utilisant $A^{[L]}$, vous pouvez calculer le coût de vos prédictions.

<a name='5'></a>
## 5 - Fonction de coût

Vous pouvez maintenant implémenter la propagation avant et la rétropropagation. Vous devez calculer le coût afin de vérifier si votre modèle apprend réellement.

<a name='ex-6'></a>
### Exercice 6 - compute_cost
Calculez le coût d'entropie croisée $J$, en utilisant la formule suivante : $$-\frac{1}{m} \sum\limits_{i = 1}^{m} (y^{(i)}\log\left(a^{[L] (i)}\right) + (1-y^{(i)})\log\left(1- a^{[L](i)}\right)) \tag{7}$$

In [ ]:
# FONCTION : compute_cost
def compute_cost(AL, Y):
    """ Implémente la fonction de coût définie par l'équation (7). Arguments : AL -- vecteur de probabilités correspondant à vos prédictions d'étiquettes, de forme (1, nombre d'exemples) Y -- vecteur d'étiquettes réelles (par exemple : contenant 0 si non-chat, 1 si chat), de forme (1, nombre d'exemples) Retourne : coût -- coût d'entropie croisée """
    
    m = Y.shape[1]

    # Calculer la perte à partir de AL et y. # (≈ 1 ligne de code) # coût = ... # VOTRE CODE COMMENCE ICI # VOTRE CODE SE TERMINE ICI coût = np.squeeze(coût) # Pour s'assurer que la forme du coût est celle attendue (ex. transforme [[17]] en 17).

    
    return cost

In [ ]:
t_Y, t_AL = compute_cost_test_case()
t_cost = compute_cost(t_AL, t_Y)

print("Cost: " + str(t_cost))

compute_cost_test(compute_cost)

**Sortie attendue :**
<table>
 <tr>
 <td><b>coût</b> </td>
 <td> 0.2797765635793422</td>  </tr>
</table>

<a name='6'></a>
## 6 - Module de rétropropagation

Comme pour la propagation avant, vous allez implémenter des fonctions auxiliaires pour la rétropropagation dans ce notebook. Rappelez-vous que la rétropropagation sert à calculer le gradient de la fonction de perte par rapport aux paramètres. 
**Rappel :** <img src="images/backprop_kiank.png" style="width:650px;height:250px;">
<caption><center><font color='purple'><b>Figure 3</b> : Propagation avant et arrière pour LINÉAIRE->RELU->LINÉAIRE->SIGMOÏDE <br> <i>Les blocs violets représentent la propagation avant, et les blocs rouges représentent la rétropropagation.</i></font></center></caption>


<!-- Pour ceux d'entre vous qui maîtrisent le calcul différentiel (ce qui n'est pas nécessaire pour ce notebook !), la règle de la chaîne peut être utilisée pour dériver la dérivée de la perte $\mathcal{L}$ par rapport à $z^{[1]}$ dans un réseau à 2 couches comme suit :

$$\frac{d \mathcal{L}(a^{[2]},y)}{{dz^{[1]}}} = \frac{d\mathcal{L}(a^{[2]},y)}{{da^{[2]}}}\frac{{da^{[2]}}}{{dz^{[2]}}}\frac{{dz^{[2]}}}{{da^{[1]}}}\frac{{da^{[1]}}}{{dz^{[1]}}} \tag{8} $$

Pour calculer le gradient $dW^{[1]} = \frac{\partial L}{\partial W^{[1]}}$, utilisez la règle de chaîne précédente : $dW^{[1]} = dz^{[1]} \times \frac{\partial z^{[1]} }{\partial W^{[1]}}$. Pendant la rétropropagation, à chaque étape, vous multipliez votre gradient courant par le gradient de la couche correspondante pour obtenir le gradient recherché.

De manière équivalente, pour calculer le gradient $db^{[1]} = \frac{\partial L}{\partial b^{[1]}}$, vous utilisez la règle de chaîne précédente : $db^{[1]} = dz^{[1]} \times \frac{\partial z^{[1]} }{\partial b^{[1]}}$.

C'est pourquoi on parle de **rétropropagation**.
!-->

De façon analogue à la propagation avant, vous allez construire la rétropropagation en trois étapes :
1. linear_backward
2. LINÉAIRE -> ACTIVATION arrière, où ACTIVATION calcule la dérivée de l'activation ReLU ou sigmoïde
3. [LINÉAIRE -> RELU] $\times$ (L-1) -> LINÉAIRE -> SIGMOÏDE arrière (modèle complet)

Pour le prochain exercice, rappelez-vous que :

- `b` est une matrice (np.ndarray) avec 1 colonne et n lignes, par ex. : b = [[1.0], [2.0]] (rappel : `b` est une constante)
- `np.sum` effectue une somme sur les éléments d'un ndarray
- `axis=1` ou `axis=0` indique si la somme est effectuée par lignes ou par colonnes, respectivement
- `keepdims` indique si les dimensions originales de la matrice doivent être conservées
- Voici un exemple illustratif :

In [ ]:
A = np.array([[1, 2], [3, 4]])

print('axis=1 and keepdims=True')
print(np.sum(A, axis=1, keepdims=True))
print('axis=1 and keepdims=False')
print(np.sum(A, axis=1, keepdims=False))
print('axis=0 and keepdims=True')
print(np.sum(A, axis=0, keepdims=True))
print('axis=0 and keepdims=False')
print(np.sum(A, axis=0, keepdims=False))

<a name='6-1'></a>
### 6.1 - Propagation linéaire arrière

Pour la couche $l$, la partie linéaire est : $Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$ (suivie d'une activation).

Supposons que vous ayez déjà calculé la dérivée $dZ^{[l]} = \frac{\partial \mathcal{L} }{\partial Z^{[l]}}$. Vous souhaitez obtenir $(dW^{[l]}, db^{[l]}, dA^{[l-1]})$.

<img src="images/linearback_kiank.png" style="width:250px;height:300px;">
<caption><center><font color='purple'><b>Figure 4</b></font></center></caption>

Les trois sorties $(dW^{[l]}, db^{[l]}, dA^{[l-1]})$ sont calculées en utilisant l'entrée $dZ^{[l]}$.

Voici les formules dont vous avez besoin :
$$ dW^{[l]} = \frac{\partial \mathcal{J} }{\partial W^{[l]}} = \frac{1}{m} dZ^{[l]} A^{[l-1] T} \tag{8}$$
$$ db^{[l]} = \frac{\partial \mathcal{J} }{\partial b^{[l]}} = \frac{1}{m} \sum_{i = 1}^{m} dZ^{[l](i)}\tag{9}$$
$$ dA^{[l-1]} = \frac{\partial \mathcal{L} }{\partial A^{[l-1]}} = W^{[l] T} dZ^{[l]} \tag{10}$$


$A^{[l-1] T}$ est la transposée de $A^{[l-1]}$.

<a name='ex-7'></a>
### Exercice 7 - linear_backward 
Utilisez les 3 formules ci-dessus pour implémenter `linear_backward()`.

**Indice :**

- En numpy, vous pouvez obtenir la transposée d'un ndarray `A` en utilisant `A.T` ou `A.transpose()`

In [ ]:
# FONCTION : linear_backward
def linear_backward(dZ, cache):
    """ Implémente la partie linéaire de la rétropropagation pour une seule couche (couche l) Arguments : dZ -- Gradient du coût par rapport à la sortie linéaire (de la couche actuelle l) cache -- tuple de valeurs (A_prev, W, b) provenant de la propagation avant dans la couche actuelle Retourne : dA_prev -- Gradient du coût par rapport à l'activation (de la couche précédente l-1), même forme que A_prev dW -- Gradient du coût par rapport à W (couche actuelle l), même forme que W db -- Gradient du coût par rapport à b (couche actuelle l), même forme que b """
    A_prev, W, b = cache
    m = A_prev.shape[1]

    ### DÉBUT DU CODE ### (≈ 3 lignes de code) # dW = ... # db = ... somme des lignes de dZ avec keepdims=True # dA_prev = ... # VOTRE CODE COMMENCE ICI 
    # VOTRE CODE SE TERMINE ICI
    return dA_prev, dW, db

In [ ]:
t_dZ, t_linear_cache = linear_backward_test_case()
t_dA_prev, t_dW, t_db = linear_backward(t_dZ, t_linear_cache)

print("dA_prev: " + str(t_dA_prev))
print("dW: " + str(t_dW))
print("db: " + str(t_db))

linear_backward_test(linear_backward)

**Sortie attendue :**
```
dA_prev: [[-1.15171336 0.06718465 -0.3204696 2.09812712]
 [ 0.60345879 -3.72508701 5.81700741 -3.84326836]
 [-0.4319552 -1.30987417 1.72354705 0.05070578]
 [-0.38981415 0.60811244 -1.25938424 1.47191593]
 [-2.52214926 2.67882552 -0.67947465 1.48119548]]
dW: [[ 0.07313866 -0.0976715 -0.87585828 0.73763362 0.00785716]
 [ 0.85508818 0.37530413 -0.59912655 0.71278189 -0.58931808]
 [ 0.97913304 -0.24376494 -0.08839671 0.55151192 -0.10290907]]
db: [[-0.14713786]
 [-0.11313155]
 [-0.13209101]]
 ```

<a name='6-2'></a>
### 6.2 - Propagation linéaire-activation arrière

Ensuite, vous allez créer une fonction qui combine les deux fonctions auxiliaires : **`linear_backward`** et l'étape arrière pour l'activation **`linear_activation_backward`**. 
Pour vous aider à implémenter `linear_activation_backward`, deux fonctions de rétropropagation sont fournies :
- **`sigmoid_backward`** : implémente la rétropropagation pour l'unité SIGMOID. Vous pouvez l'appeler ainsi :

```python
dZ = sigmoid_backward(dA, activation_cache)
```

- **`relu_backward`** : implémente la rétropropagation pour l'unité RELU. Vous pouvez l'appeler ainsi :

```python
dZ = relu_backward(dA, activation_cache)
```

Si $g(.)$ est la fonction d'activation, `sigmoid_backward` et `relu_backward` calculent $$dZ^{[l]} = dA^{[l]} * g'(Z^{[l]}). \tag{11}$$ 
<a name='ex-8'></a>
### Exercice 8 - linear_activation_backward

Implémentez la rétropropagation pour la couche *LINÉAIRE->ACTIVATION*.

In [ ]:
# FONCTION : linear_activation_backward
def linear_activation_backward(dA, cache, activation):
    """
 Implémente la rétropropagation pour la couche LINÉAIRE->ACTIVATION.
 
 Arguments :
 dA -- gradient post-activation pour la couche actuelle l
 cache -- tuple de valeurs (linear_cache, activation_cache) stocké pour calculer la rétropropagation efficacement
 activation -- la fonction d'activation à utiliser dans cette couche, sous forme de chaîne de caractères : "sigmoid" ou "relu"
 
 Retourne :
 dA_prev -- Gradient du coût par rapport à l'activation (de la couche précédente l-1), même forme que A_prev
 dW -- Gradient du coût par rapport à W (couche actuelle l), même forme que W
 db -- Gradient du coût par rapport à b (couche actuelle l), même forme que b
 """
    linear_cache, activation_cache = cache
    
    if activation == "relu":
        #(≈ 2 lignes de code) # dZ = ... # dA_prev, dW, db = ... # VOTRE CODE COMMENCE ICI 
        # VOTRE CODE SE TERMINE ICI
    elif activation == "sigmoid":
        #(≈ 2 lignes de code) # dZ = ... # dA_prev, dW, db = ... # VOTRE CODE COMMENCE ICI 
        # VOTRE CODE SE TERMINE ICI
    return dA_prev, dW, db

In [ ]:
t_dAL, t_linear_activation_cache = linear_activation_backward_test_case()

t_dA_prev, t_dW, t_db = linear_activation_backward(t_dAL, t_linear_activation_cache, activation = "sigmoid")
print("Avec sigmoid : dA_prev = " + str(t_dA_prev))
print("Avec sigmoid : dW = " + str(t_dW))
print("Avec sigmoid : db = " + str(t_db))

t_dA_prev, t_dW, t_db = linear_activation_backward(t_dAL, t_linear_activation_cache, activation = "relu")
print("Avec relu : dA_prev = " + str(t_dA_prev))
print("Avec relu : dW = " + str(t_dW))
print("Avec relu : db = " + str(t_db))

linear_activation_backward_test(linear_activation_backward)

**Sortie attendue :**
```
Avec sigmoid : dA_prev = [[ 0.11017994 0.01105339]
 [ 0.09466817 0.00949723]
 [-0.05743092 -0.00576154]]
Avec sigmoid : dW = [[ 0.10266786 0.09778551 -0.01968084]]
Avec sigmoid : db = [[-0.05729622]]
Avec relu : dA_prev = [[ 0.44090989 0. ]
 [ 0.37883606 0. ]
 [-0.2298228 0. ]]
Avec relu : dW = [[ 0.44513824 0.37371418 -0.10478989]]
Avec relu : db = [[-0.20837892]]
```

<a name='6-3'></a>
### 6.3 - Rétropropagation du modèle à L couches 
Vous allez maintenant implémenter la fonction de rétropropagation pour l'ensemble du réseau ! 
Rappelez-vous que lorsque vous avez implémenté la fonction `L_model_forward`, à chaque itération vous avez stocké un cache contenant (X, W, b et z). Dans le module de rétropropagation, vous utiliserez ces variables pour calculer les gradients. Ainsi, dans la fonction `L_model_backward`, vous parcourrez toutes les couches cachées en sens inverse, en commençant par la couche $L$. À chaque étape, vous utiliserez les valeurs en cache de la couche $l$ pour rétropropager à travers cette couche. La Figure 5 ci-dessous illustre ce passage arrière. 

<img src="images/mn_backward.png" style="width:450px;height:300px;">
<caption><center><font color='purple'><b>Figure 5</b> : Passe arrière</font></center></caption>

**Initialisation de la rétropropagation** :

Pour rétropropager à travers ce réseau, vous savez que la sortie est : $A^{[L]} = \sigma(Z^{[L]})$. Votre code doit donc calculer `dAL` $= \frac{\partial \mathcal{L}}{\partial A^{[L]}}$.
Pour ce faire, utilisez cette formule (dérivée à l'aide du calcul différentiel, dont vous n'avez pas besoin de maîtriser les détails !) :
```python
dAL = - (np.divide(Y, AL) - np.divide(1 - Y, 1 - AL)) # dérivée du coût par rapport à AL
```

Vous pouvez ensuite utiliser ce gradient post-activation `dAL` pour continuer la rétropropagation. Comme montré dans la Figure 5, vous pouvez injecter `dAL` dans la fonction arrière LINÉAIRE->SIGMOÏDE que vous avez implémentée (qui utilisera les valeurs mises en cache par la fonction `L_model_forward`). 
Ensuite, vous devrez utiliser une boucle `for` pour itérer sur toutes les autres couches en utilisant la fonction arrière LINÉAIRE->RELU. Vous devez stocker chaque dA, dW et db dans le dictionnaire `grads`. Pour ce faire, utilisez cette formule : 
$$grads["dW" + str(l)] = dW^{[l]}\tag{15} $$

Par exemple, pour $l=3$, cela stockerait $dW^{[l]}$ dans `grads["dW3"]`.

<a name='ex-9'></a>
### Exercice 9 - L_model_backward

Implémentez la rétropropagation pour le modèle *[LINÉAIRE->RELU] $\times$ (L-1) -> LINÉAIRE -> SIGMOÏDE*.

In [ ]:
# FONCTION : L_model_backward
def L_model_backward(AL, Y, caches):
    """ Implémente la rétropropagation pour [LINÉAIRE->RELU] * (L-1) -> LINÉAIRE -> SIGMOÏDE Arguments : AL -- vecteur de probabilités, sortie de la propagation avant (L_model_forward()) Y -- vecteur d'étiquettes réelles (contenant 0 si non-chat, 1 si chat) caches -- liste de caches contenant : chaque cache de linear_activation_forward() avec "relu" (caches[l], for l in range(L-1), i.e. l = 0...L-2) le cache de linear_activation_forward() avec "sigmoid" (caches[L-1]) Retourne : grads -- un dictionnaire contenant les gradients grads["dA" + str(l)] = ... grads["dW" + str(l)] = ... grads["db" + str(l)] = ... """
    grads = {}
    L = len(caches) # nombre de couches
    m = AL.shape[1]
    Y = Y.reshape(AL.shape) # après cette ligne, Y a la même forme que AL
    
    # Initialisation de la rétropropagation #(1 ligne de code) # dAL = ... # VOTRE CODE COMMENCE ICI 
    # VOTRE CODE SE TERMINE ICI # Gradients de la couche L (SIGMOID -> LINÉAIRE). Entrées : "dAL, current_cache". Sorties : "grads["dAL-1"], grads["dWL"], grads["dbL"] #(environ 5 lignes) # current_cache = ... # dA_prev_temp, dW_temp, db_temp = ... # grads["dA" + str(L-1)] = ... # grads["dW" + str(L)] = ... # grads["db" + str(L)] = ... # VOTRE CODE COMMENCE ICI 
    # VOTRE CODE SE TERMINE ICI # Boucle de l=L-2 à l=0 for l in reversed(range(L-1)): # lème couche : gradients (RELU -> LINÉAIRE). # Entrées : "grads["dA" + str(l + 1)], current_cache". Sorties : "grads["dA" + str(l)] , grads["dW" + str(l + 1)] , grads["db" + str(l + 1)] #(environ 5 lignes) # current_cache = ... # dA_prev_temp, dW_temp, db_temp = ... # grads["dA" + str(l)] = ... # grads["dW" + str(l + 1)] = ... # grads["db" + str(l + 1)] = ... # VOTRE CODE COMMENCE ICI 
        # VOTRE CODE SE TERMINE ICI
    return grads

In [ ]:
t_AL, t_Y_assess, t_caches = L_model_backward_test_case()
grads = L_model_backward(t_AL, t_Y_assess, t_caches)

print("dA0 = " + str(grads['dA0']))
print("dA1 = " + str(grads['dA1']))
print("dW1 = " + str(grads['dW1']))
print("dW2 = " + str(grads['dW2']))
print("db1 = " + str(grads['db1']))
print("db2 = " + str(grads['db2']))

L_model_backward_test(L_model_backward)

**Sortie attendue :**
```
dA0 = [[ 0. 0.52257901]
 [ 0. -0.3269206 ]
 [ 0. -0.32070404]
 [ 0. -0.74079187]]
dA1 = [[ 0.12913162 -0.44014127]
 [-0.14175655 0.48317296]
 [ 0.01663708 -0.05670698]]
dW1 = [[0.41010002 0.07807203 0.13798444 0.10502167]
 [0. 0. 0. 0. ]
 [0.05283652 0.01005865 0.01777766 0.0135308 ]]
dW2 = [[-0.39202432 -0.13325855 -0.04601089]]
db1 = [[-0.22007063]
 [ 0. ]
 [-0.02835349]]
db2 = [[0.15187861]]
```

<a name='6-4'></a>
### 6.4 - Mise à jour des paramètres

Dans cette section, vous allez mettre à jour les paramètres du modèle avec la descente de gradient : 
$$ W^{[l]} = W^{[l]} - \alpha \text{ } dW^{[l]} \tag{16}$$
$$ b^{[l]} = b^{[l]} - \alpha \text{ } db^{[l]} \tag{17}$$

où $\alpha$ est le taux d'apprentissage. 
Une fois les paramètres mis à jour, stockez-les dans le dictionnaire des paramètres.

<a name='ex-10'></a>
### Exercice 10 - Mettre à jour les paramètres

Implémentez `update_parameters()` pour mettre à jour vos paramètres avec la descente de gradient.

**Instructions :**
Mettre à jour les paramètres par descente de gradient pour chaque $W^{[l]}$ et $b^{[l]}$, avec $l = 1, 2, ..., L$.

In [ ]:
# FONCTION : update_parameters
def update_parameters(params, grads, learning_rate):
    """ Mettre à jour les paramètres avec la descente de gradient Arguments : params -- dictionnaire python contenant vos paramètres grads -- dictionnaire python contenant vos gradients, sortie de L_model_backward Retourne : paramètres -- dictionnaire python contenant vos paramètres mis à jour parameters["W" + str(l)] = ... parameters["b" + str(l)] = ... """
    parameters = copy.deepcopy(params)
    L = len(parameters) // 2 # nombre de couches dans le réseau de neurones

    # Règle de mise à jour pour chaque paramètre. Utilisez une boucle for. #(≈ 2 lignes de code) for l in range(L): # parameters["W" + str(l+1)] = ... # parameters["b" + str(l+1)] = ... # VOTRE CODE COMMENCE ICI 
        # VOTRE CODE SE TERMINE ICI
    return paramètres

In [ ]:
t_parameters, grads = update_parameters_test_case()
t_parameters = update_parameters(t_parameters, grads, 0.1)

print ("W1 = "+ str(t_parameters["W1"]))
print ("b1 = "+ str(t_parameters["b1"]))
print ("W2 = "+ str(t_parameters["W2"]))
print ("b2 = "+ str(t_parameters["b2"]))

update_parameters_test(update_parameters)

**Sortie attendue :**
```
W1 = [[-0.59562069 -0.09991781 -2.14584584 1.82662008]
 [-1.76569676 -0.80627147 0.51115557 -1.18258802]
 [-1.0535704 -0.86128581 0.68284052 2.20374577]]
b1 = [[-0.04659241]
 [-1.28888275]
 [ 0.53405496]]
W2 = [[-0.55569196 0.0354055 1.32964895]]
b2 = [[-0.84610769]]
```

### Félicitations !

Vous venez d'implémenter toutes les fonctions nécessaires à la construction d'un réseau de neurones profond, notamment :

- L'utilisation d'unités non linéaires pour améliorer votre modèle
- La construction d'un réseau de neurones plus profond (avec plus d'une couche cachée)
- L'implémentation d'une classe de réseau de neurones simple d'utilisation

Dans le prochain notebook, vous combinerez toutes ces briques pour construire et entraîner deux modèles :

- Un réseau de neurones à deux couches
- Un réseau de neurones à L couches

Excellent travail !